# ROGII Wellbore Geology

## Score: 9.089

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings("ignore")

INPUT_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = INPUT_DIR / "train"
TEST_DIR = INPUT_DIR / "test"
SAMPLE_SUB = INPUT_DIR / "sample_submission.csv"

N_PARTICLES = 500
N_SEEDS_DEFAULT = 256
N_SEEDS_EASY = 128
N_SEEDS_HARD = 384
PF_HARD_GR_NAN = 0.15
N_SEEDS = N_SEEDS_DEFAULT
PF_INIT_SPREAD = 2.0

TUNE_MAX_WELLS = 120
TUNE_N_SEEDS = 24
TUNE_SCALES = (3.0, 5.0, 8.0, 12.0)
TUNE_BEAM_WEIGHTS = (0.0, 0.05, 0.1, 0.15, 0.2)
TUNE_HOLD_WEIGHTS = (0.0, 0.05, 0.1, 0.15, 0.2)

SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73, 185.5133333333342)
SELECTOR_GR_NAN_THRESHOLD = 0.08
SELECTOR_HEEL_CORR_THRESHOLD = 0.50
SELECTOR_BASE_BIN_VARIANTS = {
    0: "pf_scale_5_hold_0.2",
    1: "pf_scale_3_hold_0.15",
    2: "pf_scale_12_beam_0.2_hold_0.15",
    3: "pf_scale_5_hold_0.15",
    4: "pf_scale_5_beam_0.05_hold_0.05",
    5: "pf_scale_12_beam_0.2_hold_0.05",
}
DEFAULT_SELECTOR_BIN_VARIANTS = {
    code: SELECTOR_BASE_BIN_VARIANTS[code % 6] for code in range(24)
}
SELECTOR_BIN_VARIANTS = dict(DEFAULT_SELECTOR_BIN_VARIANTS)
SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),
    (10, 8.0, 64.0, 2),
    (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5),
    (20, 4.0, 36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    (20, 30.0, 200.0, 2),
    (15, 10.0, 80.0, 4),
    (25, 6.0, 50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2),
    (10, 50.0, 400.0, 0),
]

BEAM_LONG_EXTRA = [
    (25, 45.0, 350.0, 1),
    (20, 35.0, 280.0, 2),
    (15, 50.0, 420.0, 0),
]

BEAM_SHORT_EXTRA = [
    (8, 6.0, 48.0, 3),
    (10, 10.0, 72.0, 4),
    (12, 5.0, 40.0, 2),
]

DELTA_CAP_PERCENTILE = 99.0
DELTA_CAP_FLOOR = 1.5
DELTA_CAP_CEIL = 4.0
MAX_DELTA_TVT = 2.5

TRAIN_WELLS = set(
    p.name.split("__")[0] for p in TRAIN_DIR.glob("*__horizontal_well.csv")
)
print(f"Train wells: {len(TRAIN_WELLS)}")

In [ ]:
def load_well(wid: str, split: str = "train") -> tuple[pd.DataFrame, pd.DataFrame]:
    base = TRAIN_DIR if split == "train" else TEST_DIR
    hw = pd.read_csv(base / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{wid}__typewell.csv")
    return hw, tw


def tvt_from_contacts(hw_tr: pd.DataFrame, tw_tr: pd.DataFrame, ref_col: str = "EGFDU") -> pd.Series:
    tw_g = tw_tr.dropna(subset=["Geology"])
    ref_tvt = tw_g.loc[tw_g["Geology"] == ref_col, "TVT"].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g["Geology"].iloc[0]
        ref_tvt = tw_g.loc[tw_g["Geology"] == ref_col, "TVT"].min()
    offset = (hw_tr["TVT"] - (ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr["Z"] - hw_tr[ref_col]) + offset


def run_particle_filter(hw: pd.DataFrame, tw: pd.DataFrame, n_particles: int = N_PARTICLES, seed: int = 42):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(dtype=float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(dtype=float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        return hw["TVT_input"].to_numpy(dtype=float).copy(), 0.0
    if len(kn) == 0:
        return hw["TVT_input"].to_numpy(dtype=float).copy(), -np.inf

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"])
    last_z = float(last["Z"])
    last_md = float(last["MD"])

    tw_at_k = np.interp(kn["TVT_input"].to_numpy(dtype=float), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10.0, 60.0))

    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy(dtype=float))
    dz = np.diff(tail["Z"].to_numpy(dtype=float))
    dm = np.diff(tail["MD"].to_numpy(dtype=float))
    m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    n = n_particles
    rng = np.random.default_rng(seed)
    ls = last_tvt + last_z
    pos = ls + PF_INIT_SPREAD * rng.standard_normal(n)
    rate = ir + 0.01 * rng.standard_normal(n)
    w = np.ones(n) / n

    mom, vn, pn, rp, rr, resamp = 0.998, 0.002, 0.005, 0.1, 0.001, 0.5

    md_v = ev["MD"].to_numpy(dtype=float)
    z_v = ev["Z"].to_numpy(dtype=float)
    gr_interp = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean())
    gr_v = gr_interp.to_numpy(dtype=float)[ev.index]

    out_vals = hw["TVT_input"].to_numpy(dtype=float).copy()
    res = np.empty(len(ev))
    prev_md = last_md
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_md, 1.0)
        rate = mom * rate + vn * rng.standard_normal(n)
        pos = pos + rate * dm_step + pn * rng.standard_normal(n)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100.0, tw_tvt[-1] + 100.0)
        pos = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.0))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(n) / n

        n_eff = 1.0 / (w**2).sum()
        if n_eff < resamp * n:
            cum = np.cumsum(w)
            u0 = rng.uniform(0, 1.0 / n)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(n) / n), 0, n - 1)
            pos = pos[idx] + rp * rng.standard_normal(n)
            rate = rate[idx] + rr * rng.standard_normal(n)
            w = np.ones(n) / n

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_md = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=N_PARTICLES, n_seeds=N_SEEDS):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
    out["pf_mean"] = pred_arr.mean(0)
    return out


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    moves = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    mc_arr = mc * np.array([2.0, 1.0, 0.0, 1.0, 2.0])

    bidx = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.0
    bn = 1
    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + moves[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)
        gr_e = (gv - tw_gr[ci]) ** 2 / es
        tot = bcost[:bn, None] + gr_e + mc_arr[None, :]
        tot = np.where(valid, tot, np.inf)
        ni_f = ni.flatten()
        tot_f = tot.flatten()
        vf = valid.flatten()
        ni_f = ni_f[vf]
        tot_f = tot_f[vf]
        order = np.argsort(tot_f)
        ni_s = ni_f[order]
        tot_s = tot_f[order]
        _, first = np.unique(ni_s, return_index=True)
        ni_u = ni_s[first]
        tot_u = tot_s[first]
        kept = min(bs, len(ni_u))
        top = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top = top[np.argsort(tot_u[top])]
        bidx[:kept] = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:] = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept
        result[step] = tw_tvt[bidx[0]]
    return result


def heel_gr_corr(hw: pd.DataFrame, tw: pd.DataFrame) -> float:
    kn = hw[hw["TVT_input"].notna()]
    if len(kn) < 10:
        return 0.5
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(dtype=float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(dtype=float)
    heel = kn.tail(50)
    hgr = heel["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(dtype=float)
    tvt = heel["TVT_input"].to_numpy(dtype=float)
    tw_at = np.interp(tvt, tw_tvt, tw_gr)
    valid = np.isfinite(hgr) & np.isfinite(tw_at)
    if valid.sum() < 5:
        return 0.5
    c = np.corrcoef(hgr[valid], tw_at[valid])[0, 1]
    if not np.isfinite(c):
        return 0.5
    return float(np.clip(c, 0.0, 1.0))


def beam_configs_for_well(n_eval: int) -> list:
    if n_eval > SELECTOR_N_EVAL_THRESHOLD:
        return BEAM_CONFIGS + BEAM_LONG_EXTRA
    return BEAM_CONFIGS + BEAM_SHORT_EXTRA


def run_beam_ensemble(hw, tw):
    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        return hw["TVT_input"].to_numpy(dtype=float).copy()

    if len(kn) == 0:
        return hw["TVT_input"].to_numpy(dtype=float).copy()

    last_tvt = float(kn.iloc[-1]["TVT_input"])
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(dtype=float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(dtype=float)
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(dtype=float)
    hgr = gr_all[ev.index]
    configs = beam_configs_for_well(len(ev))
    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r) for (bs, mc, es, r) in configs]
    beam_track = np.median(np.stack(beam_results, 0), axis=0)
    out = hw["TVT_input"].to_numpy(dtype=float).copy()
    out[list(ev.index)] = beam_track
    return out


def eval_start_train(hw: pd.DataFrame) -> int:
    gr_ok = hw["GR"].notna().to_numpy()
    if not gr_ok.any():
        return len(hw)
    return int(np.argmax(gr_ok))


def simulate_test_hw(hw: pd.DataFrame) -> pd.DataFrame | None:
    hw_sim = hw.copy()
    es = eval_start_train(hw_sim)
    if es <= 0:
        es = 1
    if es >= len(hw_sim):
        return None
    hw_sim.loc[es:, "TVT_input"] = np.nan
    if hw_sim["TVT_input"].notna().sum() < 5:
        return None
    return hw_sim


def encode_variant(scale: float, beam_weight: float, hold_weight: float) -> str:
    parts = [f"pf_scale_{scale:g}"]
    if beam_weight > 0:
        parts.append(f"beam_{beam_weight:g}")
    if hold_weight > 0:
        parts.append(f"hold_{hold_weight:g}")
    return "_".join(parts)


def apply_blend(scale, beam_weight, hold_weight, pf_by_scale, tvt_beam, last_known_tvt):
    base = pf_by_scale.get(f"pf_scale_{scale:g}", pf_by_scale["pf_scale_8"])
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    return (1.0 - hold_weight) * pred + hold_weight * last_known_tvt


def pf_seeds_for_well(hw: pd.DataFrame, tw=None) -> int:
    eval_mask = hw["TVT_input"].isna()
    if not eval_mask.any():
        return N_SEEDS_EASY
    z_eval = hw.loc[eval_mask, "Z"].to_numpy(dtype=float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    gr_nan = eval_gr_nan_frac(hw)
    heel_corr = heel_gr_corr(hw, tw) if tw is not None else 0.75
    hard_signals = int(z_span > SELECTOR_Z_SPAN_THRESHOLDS[1])
    hard_signals += int(gr_nan > PF_HARD_GR_NAN)
    hard_signals += int(heel_corr < SELECTOR_HEEL_CORR_THRESHOLD)
    easy = (
        z_span <= SELECTOR_Z_SPAN_THRESHOLDS[0]
        and gr_nan <= SELECTOR_GR_NAN_THRESHOLD * 0.5
        and heel_corr >= SELECTOR_HEEL_CORR_THRESHOLD
    )
    if hard_signals >= 3:
        return N_SEEDS_HARD
    if easy:
        return N_SEEDS_EASY
    return N_SEEDS_DEFAULT


def eval_gr_nan_frac(hw: pd.DataFrame) -> float:
    ev = hw["TVT_input"].isna()
    if not ev.any():
        return 0.0
    return float(hw.loc[ev, "GR"].isna().mean())


def selector_well_code(hw, tw=None):
    eval_mask = hw["TVT_input"].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, "Z"].to_numpy(dtype=float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    gr_nan = eval_gr_nan_frac(hw)
    heel_corr = heel_gr_corr(hw, tw) if tw is not None else 0.75
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side="right"))
    gr_bin = int(gr_nan > SELECTOR_GR_NAN_THRESHOLD)
    heel_bin = int(heel_corr < SELECTOR_HEEL_CORR_THRESHOLD)
    base_code = n_bin + 2 * z_bin
    code = base_code + 6 * gr_bin + 12 * heel_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span, gr_nan, heel_corr


def parse_selector_variant(name):
    parts = name.split("_")
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if "beam" in parts:
        beam_weight = float(parts[parts.index("beam") + 1])
    if "hold" in parts:
        hold_weight = float(parts[parts.index("hold") + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt, hw, tw):
    heel_corr = heel_gr_corr(hw, tw)
    scale, beam_w, hold_w = parse_selector_variant(name)
    beam_w = float(np.clip(beam_w + 0.25 * heel_corr, 0.0, 0.35))
    hold_w = float(np.clip(hold_w * (1.15 - 0.45 * heel_corr), 0.0, 0.3))
    return apply_blend(scale, beam_w, hold_w, pf_by_scale, tvt_beam, last_known_tvt)


def tune_one_well(wid: str, n_seeds: int = TUNE_N_SEEDS):
    hw, tw = load_well(wid, "train")
    if "TVT" not in hw.columns:
        return None

    hw_sim = simulate_test_hw(hw)
    if hw_sim is None:
        return None

    eval_mask = hw_sim["TVT_input"].isna() & hw["GR"].notna() & hw["TVT"].notna()
    if eval_mask.sum() < 20:
        return None

    eval_pos = np.where(eval_mask.to_numpy())[0]
    y_true = hw["TVT"].to_numpy(dtype=float)[eval_pos]

    pf_by_scale = run_pf_lik_ensemble_scales(hw_sim, tw, n_seeds=n_seeds)
    tvt_beam = run_beam_ensemble(hw_sim, tw)
    last_known = hw_sim["TVT_input"].dropna()
    last_tvt = float(last_known.iloc[-1]) if len(last_known) else float(np.nanmean(pf_by_scale["pf_scale_8"]))

    best_rmse = np.inf
    best_variant = SELECTOR_GLOBAL_VARIANT
    best_params = (8.0, 0.0, 0.2)

    for scale in TUNE_SCALES:
        for beam_w in TUNE_BEAM_WEIGHTS:
            for hold_w in TUNE_HOLD_WEIGHTS:
                pred = apply_blend(scale, beam_w, hold_w, pf_by_scale, tvt_beam, last_tvt)
                rmse = float(np.sqrt(np.mean((y_true - pred[eval_pos]) ** 2)))
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_params = (scale, beam_w, hold_w)
                    best_variant = encode_variant(scale, beam_w, hold_w)

    code, _, n_eval, z_span, gr_nan, heel_corr = selector_well_code(hw_sim, tw)
    default_variant = DEFAULT_SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    default_pred = apply_selector_variant(default_variant, pf_by_scale, tvt_beam, last_tvt, hw_sim, tw)
    default_rmse = float(np.sqrt(np.mean((y_true - default_pred[eval_pos]) ** 2)))

    return {
        "well": wid,
        "code": code,
        "base_code": code % 6,
        "n_eval": n_eval,
        "z_span": z_span,
        "gr_nan": gr_nan,
        "heel_corr": heel_corr,
        "best_variant": best_variant,
        "best_rmse": best_rmse,
        "default_rmse": default_rmse,
        "scale": best_params[0],
        "beam_w": best_params[1],
        "hold_w": best_params[2],
    }


def fit_bin_variants(tune_df: pd.DataFrame) -> dict[int, str]:
    out = dict(DEFAULT_SELECTOR_BIN_VARIANTS)
    for code, grp in tune_df.groupby("code"):
        out[int(code)] = grp["best_variant"].value_counts().index[0]
    for code in range(24):
        if code not in tune_df["code"].values:
            base = int(code % 6)
            if base in out:
                out[code] = out[base]
    return out


def fit_delta_tvt_cap(well_ids: list[str], percentile: float = DELTA_CAP_PERCENTILE) -> float:
    deltas = []
    for wid in well_ids:
        hw, _ = load_well(wid, "train")
        if "TVT" not in hw.columns:
            continue
        hw_sim = simulate_test_hw(hw)
        if hw_sim is None:
            continue
        eval_mask = hw_sim["TVT_input"].isna() & hw["TVT"].notna()
        if eval_mask.sum() < 20:
            continue
        tvt = hw["TVT"].to_numpy(dtype=float)
        pos = np.where(eval_mask.to_numpy())[0]
        dt = np.abs(np.diff(tvt[pos]))
        if len(dt):
            deltas.extend(dt.tolist())
    if not deltas:
        return MAX_DELTA_TVT
    cap = float(np.percentile(deltas, percentile))
    return float(np.clip(cap, DELTA_CAP_FLOOR, DELTA_CAP_CEIL))


def apply_delta_tvt_cap(tvt: np.ndarray, hw: pd.DataFrame, max_delta: float) -> np.ndarray:
    out = tvt.copy()
    ev = hw["TVT_input"].isna().to_numpy()
    ev_pos = np.where(ev)[0]
    if len(ev_pos) == 0:
        return out
    start = int(ev_pos[0])
    if start > 0 and np.isfinite(out[start - 1]):
        prev = float(out[start - 1])
    else:
        prev = float(out[start])
    for i in ev_pos:
        cur = float(out[i])
        cur = float(np.clip(cur, prev - max_delta, prev + max_delta))
        out[i] = cur
        prev = cur
    return out


def pick_tune_wells(well_ids: list[str], max_wells: int = TUNE_MAX_WELLS) -> list[str]:
    well_ids = sorted(well_ids)
    if len(well_ids) <= max_wells:
        return well_ids
    rng = np.random.default_rng(42)
    return sorted(rng.choice(well_ids, max_wells, replace=False).tolist())

In [ ]:
tune_wells = pick_tune_wells(sorted(TRAIN_WELLS))
MAX_DELTA_TVT = fit_delta_tvt_cap(tune_wells)
print(f"Delta TVT cap p{DELTA_CAP_PERCENTILE:g}: {MAX_DELTA_TVT:.3f} ft")
print(f"Tuning on {len(tune_wells)} / {len(TRAIN_WELLS)} train wells  seeds={TUNE_N_SEEDS}")

tune_rows = []
for i, wid in enumerate(tune_wells, 1):
    row = tune_one_well(wid, n_seeds=TUNE_N_SEEDS)
    if row is None:
        if i % 10 == 0 or i == len(tune_wells):
            print(f"  {i}/{len(tune_wells)}  skip {wid}")
        continue
    tune_rows.append(row)
    if i % 10 == 0 or i == len(tune_wells):
        print(f"  {i}/{len(tune_wells)}  {wid}  best_rmse={row['best_rmse']:.3f}")

tune_df = pd.DataFrame(tune_rows)
print(f"\nTuned wells: {len(tune_df)}")

if len(tune_df):
    tuned_variants = fit_bin_variants(tune_df)
    SELECTOR_BIN_VARIANTS.update(tuned_variants)

    tuned_global = tune_df["best_variant"].value_counts().index[0]
    SELECTOR_GLOBAL_VARIANT = tuned_global

    print("\nDefault vs tuned selector (mean eval RMSE on tune set):")
    print(f"  default: {tune_df['default_rmse'].mean():.4f}")
    print(f"  tuned:   {tune_df['best_rmse'].mean():.4f}")

    print("\nBin variants (default -> tuned, 24-bin):")
    for code in sorted(tuned_variants):
        old = DEFAULT_SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
        new = tuned_variants[code]
        n = int((tune_df["code"] == code).sum())
        mark = " *" if old != new else ""
        print(f"  bin {code:2d} (n={n:3d}): {old} -> {new}{mark}")

    print(f"\nGlobal variant: {SELECTOR_GLOBAL_VARIANT}")
else:
    print("No tune results; keeping default selector table")

In [ ]:
sample = pd.read_csv(SAMPLE_SUB)
sample["well"] = sample["id"].str[:8]
sample["row_idx"] = sample["id"].str[9:].astype(int)
TEST_WELLS = sorted(sample["well"].unique())
print(f"Submission wells: {len(TEST_WELLS)}  rows: {len(sample):,}")

rows = []
for wid in TEST_WELLS:
    print(f"Processing {wid}...")
    hw_te, tw_te = load_well(wid, "test")

    tvt_phys = None
    hw_tr = None
    tw_tr = None

    if wid in TRAIN_WELLS:
        try:
            hw_tr, tw_tr = load_well(wid, "train")
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print("  physical model OK")
        except Exception as e:
            print(f"  physical model failed: {e}")

    tw_ref = tw_tr if tw_tr is not None else tw_te

    if tvt_phys is not None:
        print("  skip PF/beam (physical model)")
        ws = sample[sample["well"] == wid]
        for _, row in ws.iterrows():
            rows.append({"id": row["id"], "tvt": float(tvt_phys.iloc[int(row["row_idx"])])})
        print(f"  added {len(ws)} rows")
        continue

    code, variant, n_eval, z_span, gr_nan, heel_corr = selector_well_code(hw_te, tw_ref)
    n_pf_seeds = pf_seeds_for_well(hw_te, tw_ref)

    try:
        pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_seeds=n_pf_seeds)
        print(f"  PF {n_pf_seeds}-seed ensemble OK")
    except Exception as e:
        print(f"  PF failed: {e}")
        last_known = hw_te["TVT_input"].dropna()
        last_val = float(last_known.iloc[-1]) if len(last_known) else 0.0
        fallback = hw_te["TVT_input"].fillna(last_val).to_numpy(dtype=float)
        pf_by_scale = {f"pf_scale_{scale:g}": fallback.copy() for scale in SELECTOR_SCALES}

    try:
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print("  beam ensemble OK")
    except Exception as e:
        print(f"  beam failed: {e}")
        tvt_beam = pf_by_scale["pf_scale_8"].copy()

    last_known = hw_te["TVT_input"].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) else float(np.nanmean(pf_by_scale["pf_scale_8"]))
    tvt_selector = apply_selector_variant(variant, pf_by_scale, tvt_beam, last_known_tvt, hw_te, tw_ref)
    tvt_selector = apply_delta_tvt_cap(tvt_selector, hw_te, MAX_DELTA_TVT)
    print(f"  selector code={code} variant={variant} n_eval={n_eval:.0f} z_span={z_span:.3f} gr_nan={gr_nan:.3f} heel={heel_corr:.2f} dcap={MAX_DELTA_TVT:.2f}")

    ws = sample[sample["well"] == wid]
    for _, row in ws.iterrows():
        ridx = int(row["row_idx"])
        if tvt_phys is not None:
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            tvt_val = float(tvt_selector[ridx])
        rows.append({"id": row["id"], "tvt": tvt_val})
    print(f"  added {len(ws)} rows")

submission = pd.DataFrame(rows)
submission = sample[["id"]].merge(submission, on="id", how="left")
missing = submission["tvt"].isna().sum()
if missing:
    raise ValueError(f"missing predictions: {missing}")

submission.to_csv("submission.csv", index=False)
print(f"\nWrote submission.csv rows={len(submission):,}")
print(submission.head())
print(f"tvt range: {submission['tvt'].min():.2f} .. {submission['tvt'].max():.2f}")